In [ ]:
#@title Check GPU
import pandas as pd, subprocess

def gpu_info():
    outv = subprocess.run([
        'nvidia-smi',
        '--query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.free',
        '--format=csv'],
        stdout=subprocess.PIPE).stdout.decode('utf-8')
    header, rec = outv.split('\n')[:-1]
    return pd.DataFrame(
        {' '.join(k.strip().split('.')).capitalize(): v
         for k, v in zip(header.split(','), rec.split(','))}, index=[0]).T

gpu_info()

In [ ]:
#@title Install Dependencies
# @markdown Run once per new runtime — will auto-restart.
import warnings
probably_using_colab = False
try:
    import google
    probably_using_colab = True
except ImportError:
    warnings.warn("Assuming local runtime")

try:
    import keyframed
except ImportError:
    if probably_using_colab:
        %pip install ftfy einops braceexpand requests transformers clip open_clip_torch omegaconf pytorch-lightning==1.9.4 kornia k-diffusion ninja safetensors noise
        %pip install -U git+https://github.com/huggingface/huggingface_hub
        %pip install napm keyframed
    else:
        %pip install -r klmc2/requirements.txt
        %pip install safetensors noise
    exit()

In [ ]:
# @markdown # Setup Workspace { display-mode: "form" }
import os
from pathlib import Path
import warnings

probably_using_colab = False
try:
    import google
    if Path('/content').exists():
        probably_using_colab = True
except ImportError:
    warnings.warn("Assuming local runtime")

mount_gdrive = True  # @param {type:'boolean'}

outdir = Path('./frames')
if not os.environ.get('XDG_CACHE_HOME'):
    os.environ['XDG_CACHE_HOME'] = str(Path('~/.cache').expanduser())

if mount_gdrive and probably_using_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    Path('/content/drive/MyDrive/AI/models/.cache/').mkdir(parents=True, exist_ok=True)
    os.environ['XDG_CACHE_HOME'] = '/content/drive/MyDrive/AI/models/.cache'
    outdir = Path('/content/drive/MyDrive/AI/klmc2/frames/')

outdir.mkdir(parents=True, exist_ok=True)
debug_dir = outdir.parent / 'debug_frames'
debug_dir.mkdir(parents=True, exist_ok=True)

os.environ['NAPM_PATH'] = str(Path(os.environ['XDG_CACHE_HOME']) / 'napm')
Path(os.environ['NAPM_PATH']).mkdir(parents=True, exist_ok=True)

import napm
napm.pseudoinstall_git_repo('https://github.com/Stability-AI/stablediffusion', add_install_dir_to_path=True)

models_path = "/content/models" if probably_using_colab else os.environ['XDG_CACHE_HOME']
if mount_gdrive and probably_using_colab:
    models_path = "/content/drive/MyDrive/AI/models"
Path(models_path).mkdir(parents=True, exist_ok=True)

In [ ]:
# @markdown # Imports and Definitions { display-mode: "form" }
import napm
from ldm.util import instantiate_from_config
from base64 import b64encode
from collections import defaultdict
from concurrent import futures
from functools import partial
from io import BytesIO
from pathlib import Path
import base64, math, matplotlib.image, random, re, requests, sys, time, warnings
from requests.exceptions import HTTPError
from urllib.parse import urlparse

import functorch, huggingface_hub
from IPython.display import display, Video, HTML
import k_diffusion as K
from keyframed import Curve, ParameterGroup, SmoothCurve
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
from PIL import Image
import torch
from torch import nn
from tqdm.auto import tqdm, trange
from loguru import logger
from natsort import natsorted

cpu = torch.device("cpu")
device = torch.device("cuda")

# ── Utilities ──────────────────────────────────────────────────────────────────

def get_latest_frame(i=None, latest_frame_fpath=None):
    if latest_frame_fpath is not None:
        latest_frame = latest_frame_fpath
    elif i is None:
        frames = list(Path('frames').glob("*.png"))
        latest_frame = natsorted(frames)[-1]
    else:
        latest_frame = Path('frames') / f"out_{i:05}.png"
    img = matplotlib.image.imread(latest_frame)
    return np.flip(img, axis=0)

def plot_prompts(prompts=None, n=1000, settings=None, **kw):
    if prompts:
        for p in prompts: p.weight.plot(n=n, **kw)

def plot_param(param, settings=None, prompts=None, n=1000, **kw):
    settings.parameters[param].plot(n=n, **kw)

@logger.catch
def write_debug_frame_at_(i=None, n=300, prompts=None, stuff_to_plot=None,
                           latest_frame_fpath=None, pil_image=None, settings=None):
    if not stuff_to_plot:
        return
    plotting_funcs = {
        'prompts': plot_prompts,
        **{p: partial(plot_param, param=p) for p in ('g','h','sigma','gamma','alpha','tau')}
    }
    test_im = pil_image or get_latest_frame(i, latest_frame_fpath)
    fig = plt.figure()
    ax_objs = fig.subplots(len(stuff_to_plot), 1, sharex=True)
    h_px, w_px = test_im.size
    fig.set_size_inches(h_px / fig.dpi, w_px / fig.dpi)
    buffer = BytesIO()
    for j, cat in enumerate(stuff_to_plot):
        ax = ax_objs if len(stuff_to_plot) == 1 else ax_objs[j]
        plt.sca(ax); plt.tight_layout(); plt.axis('off')
        plotting_funcs[cat](prompts=prompts, settings=settings, n=n, zorder=1)
        plt.axvline(x=i)
    fig.savefig(buffer, transparent=True)
    plt.close()
    buffer.seek(0)
    plot_pil = Image.open(buffer)
    debug_im_path = debug_dir / f"debug_out_{i:05}.png"
    test_im = test_im.convert('RGBA')
    test_im.paste(plot_pil, (0, 0), plot_pil)
    test_im.save(debug_im_path)
    buffer.close()
    return test_im, plot_pil

def show_video(video_path, video_width=512):
    return display(Video(video_path, width=video_width))

if probably_using_colab:
    def show_video(video_path, video_width=512):
        data = open(video_path, "r+b").read()
        url = f"data:video/mp4;base64,{b64encode(data).decode()}"
        return display(HTML(f'<video width={video_width} controls><source src="{url}"></video>'))

# ── Prompt and curve helpers ───────────────────────────────────────────────────

class Prompt:
    def __init__(self, text, weight_schedule):
        self.text = text
        self.encoded = sd_model.get_learned_conditioning([text])
        self.weight = SmoothCurve(weight_schedule)

def handle_chigozienri_curve_format(v):
    return v[1:-1] if v.startswith('(') and v.endswith(')') else v

def parse_curve_string(txt, f=float):
    sched = {}
    for tok in txt.split(','):
        k, v = tok.split(':')
        sched[int(k)] = f(handle_chigozienri_curve_format(v))
    return sched

def parse_curvable_string(param, is_int=False):
    if isinstance(param, dict): return param
    f = int if is_int else float
    try: return f(param)
    except ValueError: return parse_curve_string(txt=param, f=f)

# ── Image init helpers ─────────────────────────────────────────────────────────

def generate_perlin_init(height, width, octaves=6, scale=0.05, seed=None):
    """Neutral structured init: all spatial frequencies, no semantic bias."""
    try:
        from noise import pnoise2
        rng = random.Random(seed)
        offsets = [rng.uniform(0, 100) for _ in range(3)]
        img = np.zeros((height, width, 3), dtype=np.float32)
        for c in range(3):
            for y in range(height):
                for x in range(width):
                    img[y, x, c] = pnoise2(x*scale+offsets[c], y*scale+offsets[c], octaves=octaves)
    except ImportError:
        warnings.warn("noise package unavailable, using numpy approximation")
        rng = np.random.default_rng(seed)
        img = np.zeros((height, width, 3), dtype=np.float32)
        for c in range(3):
            for oct_ in range(octaves):
                freq, amp = 2**oct_, 0.5**oct_
                ph = rng.uniform(0, 2*np.pi, (2,))
                xs = np.linspace(0, freq*np.pi*scale*width, width)
                ys = np.linspace(0, freq*np.pi*scale*height, height)
                img[:, :, c] += amp * np.outer(np.sin(ys+ph[0]), np.sin(xs+ph[1]))
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return Image.fromarray((img * 255).astype(np.uint8))

def load_init_image(init_image, height, width):
    if isinstance(init_image, Image.Image):
        x_pil = init_image.resize([width, height])
    else:
        if not Path(str(init_image)).exists():
            raise FileNotFoundError(f"Init image not found: {init_image}")
        x_pil = Image.open(init_image).resize([width, height])
    x_np = np.array(x_pil.convert('RGB')).astype(np.float16) / 255.0
    x = torch.from_numpy((2.*x_np[None].transpose(0,3,1,2) - 1.)).to('cuda')
    return x

def save_image_fn(image, name, i, n, prompts=None, settings=None, stuff_to_plot=None):
    pil_image = K.utils.to_pil_image(image)
    if i % 10 == 0 or i == n - 1:
        print(f'\nIteration {i}/{n}:')
        display(pil_image)
    if i == n - 1: print('\nDone!')
    pil_image.save(name)
    if stuff_to_plot:
        _, debug_plot = write_debug_frame_at_(
            i=i, n=n, prompts=prompts, settings=settings,
            stuff_to_plot=stuff_to_plot, pil_image=pil_image)
        if i % 10 == 0 or i == n - 1:
            display(debug_plot)

# ── Checkpointing ──────────────────────────────────────────────────────────────

def write_klmc2_state(**state):
    obj = {k: (v.clone().detach().cpu() if hasattr(v,'detach') else v) for k,v in state.items()}
    fpath = Path(outdir) / f"klmc2_state_{state.get('i',0):05}.ckpt"
    with open(fpath, 'wb') as f: torch.save(obj, f=f)

def read_klmc2_state(root=None, latest_frame=-1):
    root = root or outdir
    checkpoints = natsorted([str(p) for p in Path(root).glob("*.ckpt")])
    if not checkpoints: return None
    if latest_frame < 0:
        ckpt = checkpoints[-1]
    else:
        ckpt = checkpoints[0]
        for fname in checkpoints:
            if int(re.findall(r'([0-9]+).ckpt', fname)[0]) <= latest_frame:
                ckpt = fname
    with open(ckpt, 'rb') as f:
        return torch.load(f=f, map_location='cuda')

# ── Denoiser ───────────────────────────────────────────────────────────────────

BETA = torch.tensor(0.99, device=device)
DAMPING_POWMH = torch.tensor(0.01, device=device)

class NormalizingCFGDenoiser(nn.Module):
    def __init__(self, model, g):
        super().__init__()
        self.inner_model = model
        self.g = g
        self.eps_norms = defaultdict(lambda: (0, 0))

    def mean_sq(self, x):
        return x.pow(2).flatten(1).mean(1)

    @torch.no_grad()
    def update_eps_norm(self, eps, sigma):
        sigma = sigma[0].item()
        eps_norm = self.mean_sq(eps).mean()
        avg, count = self.eps_norms[sigma]
        avg = avg*count/(count+1) + eps_norm/(count+1)
        self.eps_norms[sigma] = (avg, count+1)
        return avg

    def forward(self, x, sigma, uncond, cond, g):
        x_in = torch.cat([x]*2)
        sigma_in = torch.cat([sigma]*2)
        cond_in = torch.cat([uncond, cond])
        denoised = self.inner_model(x_in, sigma_in, cond=cond_in)
        eps = K.sampling.to_d(x_in, sigma_in, denoised)
        eps_uc, eps_c = eps.chunk(2)
        eps_norm = self.update_eps_norm(eps, sigma).sqrt()
        c = eps_c - eps_uc
        cond_scale = g * eps_norm / self.mean_sq(c).sqrt()
        eps_final = eps_uc + c * K.utils.append_dims(cond_scale, x.ndim)
        return x - eps_final * K.utils.append_dims(sigma, eps.ndim)

# ── HVP methods ────────────────────────────────────────────────────────────────

def powmh(x, power, damping=DAMPING_POWMH):
    vals, vecs = torch.linalg.eigh(x)
    vals = vals.abs().add(damping).pow(power)
    return torch.einsum("...ab,...b,...cb->...ac", vecs, vals, vecs)

def _grad_fn(model, x, sigma, alpha, extra_args):
    s_in = x.new_ones([x.shape[0]])
    return (x - model(x, sigma*s_in, **extra_args)) + alpha*x

def hvp_fn_forward_functorch(model, x, sigma, v, alpha, extra_args, hvp_state=None, **_):
    def gf(x, sigma): return _grad_fn(model, x, sigma, alpha, extra_args)
    jvp_fn = lambda v: functorch.jvp(gf, (x, sigma), (v, torch.zeros_like(sigma)))
    grad, jvp_out = functorch.vmap(jvp_fn)(v)
    return grad[0], jvp_out, None

def hvp_fn_reverse(model, x, sigma, v, alpha, extra_args, hvp_state=None, **_):
    vjps = []
    with torch.enable_grad():
        x_ = x.clone().requires_grad_()
        grad = _grad_fn(model, x_, sigma, alpha, extra_args)
        for k, item in enumerate(v):
            vjps.append(torch.autograd.grad(grad, x_, item, retain_graph=k<len(v)-1)[0])
    return grad.detach(), torch.stack(vjps), None

def hvp_fn_zero(model, x, sigma, v, alpha, extra_args, hvp_state=None, **_):
    return _grad_fn(model, x, sigma, alpha, extra_args), torch.zeros_like(v), None

def hvp_fn_fake(model, x, sigma, v, alpha, extra_args, hvp_state=None, **_):
    return _grad_fn(model, x, sigma, alpha, extra_args), (1+alpha)*v, None

def hvp_fn_kronecker(model, x, sigma, v, alpha, extra_args, hvp_state, k, beta, h, **_):
    mean, kron_1, kron_2, kron_3 = [hvp_state[k_] for k_ in ('mean','kron_1','kron_2','kron_3')]
    grad = _grad_fn(model, x, sigma, alpha, extra_args)
    with torch.cuda.amp.autocast(dtype=torch.float32), K.utils.tf32_mode(matmul=False):
        _, d1, d2, d3 = x.shape
        beta = torch.exp(-k * h)
        mean.mul_(beta).add_(grad, alpha=1-beta)
        kron_1.mul_(beta).add_(torch.einsum('nabc,nzbc->naz', grad, grad), alpha=(1-beta)/(d2*d3))
        kron_2.mul_(beta).add_(torch.einsum('nabc,nazc->nbz', grad, grad), alpha=(1-beta)/(d1*d3))
        kron_3.mul_(beta).add_(torch.einsum('nabc,nabz->ncz', grad, grad), alpha=(1-beta)/(d1*d2))
        cov_1 = kron_1 - torch.einsum('nabc,nzbc->naz', mean, mean)/(d2*d3)
        cov_2 = kron_2 - torch.einsum('nabc,nazc->nbz', mean, mean)/(d1*d3)
        cov_3 = kron_3 - torch.einsum('nabc,nabz->ncz', mean, mean)/(d1*d2)
        hvp = torch.einsum('...nabc,nad,nbe,ncf->...ndef',
                           v, powmh(cov_1,1/2), powmh(cov_2,1/2), powmh(cov_3,1/2))
        hvp_state.update({'mean':mean,'kron_1':kron_1,'kron_2':kron_2,'kron_3':kron_3})
        return grad, hvp, hvp_state

HVP_FNS = {
    'forward-functorch': hvp_fn_forward_functorch,
    'reverse':           hvp_fn_reverse,
    'zero':              hvp_fn_zero,
    'fake':              hvp_fn_fake,
    'kronecker':         hvp_fn_kronecker,
}

# ── Multicond HVP ──────────────────────────────────────────────────────────────

def multicond_hvp(model, x, sigma, v, alpha, extra_args, prompts, hvp_fn, i, hvp_state,
                  k=None, beta=None, h=None):
    grad = h2_v = h2_noise_v2 = h2_noise_x2 = torch.zeros_like(x)
    wt_norm = 0
    for prompt in prompts:
        wt = prompt.weight[i]
        if wt == 0: continue
        wt_norm += wt
        wt = torch.tensor(wt, device=x.device)
        extra_args['cond'] = prompt.encoded
        g_, (hv_, hnv2_, hnx2_), hvp_state = hvp_fn(
            model=model, x=x, sigma=sigma, v=v, alpha=alpha,
            extra_args=extra_args, hvp_state=hvp_state, k=k, beta=beta, h=h)
        grad       = grad       + g_   * wt
        h2_v       = h2_v       + hv_  * wt
        h2_noise_v2 = h2_noise_v2 + hnv2_ * wt
        h2_noise_x2 = h2_noise_x2 + hnx2_ * wt
    grad/=wt_norm; h2_v/=wt_norm; h2_noise_v2/=wt_norm; h2_noise_x2/=wt_norm
    return grad, h2_v, h2_noise_v2, h2_noise_v2, h2_noise_x2, hvp_state

# ── KLMC2 step ─────────────────────────────────────────────────────────────────

def klmc2_step(model, prompts, x, v, h, gamma, alpha, tau, g, sigma, sigmas, steps,
               hvp_method, i, callback, extra_args, hvp_state, k, beta):
    psi_0 = lambda g,t: torch.exp(-g*t)
    psi_1 = lambda g,t: -torch.expm1(-g*t)/g
    psi_2 = lambda g,t: (torch.expm1(-g*t)+g*t)/g**2
    def phi_2(g,t_):
        t=t_.double(); return ((torch.exp(-g*t)*(torch.expm1(g*t)-g*t))/g**2).to(t_)
    def phi_3(g,t_):
        t=t_.double(); return ((torch.exp(-g*t)*(2+g*t+torch.exp(g*t)*(g*t-2)))/g**3).to(t_)

    xs = torch.linspace(0, h, 1001, device=x.device)
    ys = [f(gamma, xs) for f in (psi_0, psi_1, phi_2, phi_3)]
    cov = torch.tensor([[torch.trapz(ys[a]*ys[b], x=xs) for b in range(4)] for a in range(4)],
                       device=x.device)
    noise_v, noise_x, noise_v2, noise_x2 =         torch.distributions.MultivariateNormal(x.new_zeros([4]), cov).sample(x.shape).unbind(-1)

    extra_args['g'] = g
    grad, h2_v, h2_noise_v2, h2_noise_v2, h2_noise_x2, hvp_state = multicond_hvp(
        model=model, x=x, sigma=sigma,
        v=torch.stack([v, noise_v2, noise_x2]),
        alpha=alpha, extra_args=extra_args, prompts=prompts,
        hvp_fn=HVP_FNS[hvp_method], i=i, hvp_state=hvp_state, k=k, beta=beta, h=h)

    x_ref, old_den = x, None
    for j in range(len(sigmas)-1):
        den = (x_ref - grad) if j==0 else model(x_ref, sigmas[j]*x.new_ones([x.shape[0]]), **extra_args)
        dt = sigmas[j+1] - sigmas[j]
        if old_den is None or sigmas[j+1]==0:
            x_ref = x_ref + K.sampling.to_d(x_ref, sigmas[j], den)*dt
        else:
            h_o = sigmas[j].log()-sigmas[j+1].log()
            h_l = sigmas[j-1].log()-sigmas[j].log()
            den_d = (1+h_o/(2*h_l))*den - (h_o/(2*h_l))*old_den
            x_ref = x_ref + K.sampling.to_d(x_ref, sigmas[j], den_d)*dt
        old_den = den
    if callback: callback({'i':i, 'denoised':x_ref})

    ns = (2*gamma*tau*sigma**2).sqrt()
    v_n = psi_0(gamma,h)*v - psi_1(gamma,h)*grad - phi_2(gamma,h)*h2_v + ns*(noise_v-h2_noise_v2)
    x_n = x + psi_1(gamma,h)*v - psi_2(gamma,h)*grad - phi_3(gamma,h)*h2_v + ns*(noise_x-h2_noise_x2)
    return x_n, v_n, grad, hvp_state

# ── Main sampler ───────────────────────────────────────────────────────────────

@torch.no_grad()
def sample_mcmc_klmc2(sd_model, init_image, height, width, n, hvp_method='kronecker',
                      prompts=None, settings=None, resume=False, resume_from=-1,
                      img_init_steps=None, stuff_to_plot=None):
    if stuff_to_plot is None: stuff_to_plot = ['prompts','h']
    torch.cuda.empty_cache()
    wrappers = {'eps': K.external.CompVisDenoiser, 'v': K.external.CompVisVDenoiser}
    g0 = settings[0]['g']
    model_wrap = wrappers[sd_model.parameterization](sd_model)
    model_wrap_cfg = NormalizingCFGDenoiser(model_wrap, g0)
    sigma_min, sigma_max = model_wrap.sigmas[0].item(), model_wrap.sigmas[-1].item()
    uc = sd_model.get_learned_conditioning([''])
    extra_args = {'uncond': uc, 'g': g0}
    sigma0 = settings[0]['sigma']

    with torch.cuda.amp.autocast(), futures.ThreadPoolExecutor() as ex:
        def callback(info):
            rgb = sd_model.decode_first_stage(info['denoised'])
            ex.submit(save_image_fn, image=rgb, name=(outdir/f"out_{info['i']:05}.png"),
                      i=info['i'], n=n, prompts=prompts, settings=settings, stuff_to_plot=stuff_to_plot)

        x = None
        if init_image:
            x = load_init_image(init_image, height, width)
            x = sd_model.get_first_stage_encoding(sd_model.encode_first_stage(x))

        i_resume, v = 0, None
        if resume:
            state = read_klmc2_state(latest_frame=resume_from)
            if state:
                x, v, i_resume = state['x'], state['v'], state['i']

        if x is None:
            extra_args['cond'] = prompts[0].encoded
            x = torch.randn([1,4,height//8,width//8], device=device)*sigma_max
            sp = K.sampling.get_sigmas_karras(img_init_steps, sigma0, sigma_max, device=device)[:-1]
            x = K.sampling.sample_dpmpp_sde(model_wrap_cfg, x, sp, extra_args=extra_args)

        if v is None: v = torch.randn_like(x)*sigma0

        hvp_state = dict(
            mean  = torch.zeros_like(x),
            kron_1= torch.zeros([x.shape[0],x.shape[1],x.shape[1]], device=device),
            kron_2= torch.zeros([x.shape[0],x.shape[2],x.shape[2]], device=device),
            kron_3= torch.zeros([x.shape[0],x.shape[3],x.shape[3]], device=device),
        )

        for i in trange(n):
            if resume and i < i_resume: continue
            s = settings[i]
            h     = torch.tensor(s['h'],     device=device)
            gamma = torch.tensor(s['gamma'], device=device)
            alpha = torch.tensor(s['alpha'], device=device)
            tau   = torch.tensor(s['tau'],   device=device)
            sigma = torch.tensor(s['sigma'], device=device)
            k_t   = torch.tensor(s['k'],     device=device)
            g     = s['g']
            sigmas = K.sampling.get_sigmas_karras(int(s['steps']), sigma_min, sigma.item(), device=device)[:-1]
            x, v, grad, hvp_state = klmc2_step(
                model_wrap_cfg, prompts, x, v, h, gamma, alpha, tau, g,
                sigma, sigmas, int(s['steps']), hvp_method, i, callback,
                extra_args, hvp_state, k=k_t, beta=None)
            if i % 10 == 0:
                ex.submit(write_klmc2_state, v=v, x=x, i=i)

In [ ]:
#@title Select and Load Model
import napm
from ldm.util import instantiate_from_config

model_checkpoint = "v1-5-pruned-emaonly.ckpt" #@param ["sd-v1-4.ckpt", "v1-5-pruned-emaonly.ckpt", "v2-1_768-ema-pruned.ckpt", "v2-1_512-ema-pruned.ckpt", "robo-diffusion-v1.ckpt", "model-epoch05-float16.ckpt", "custom_hf", "custom_path"]

# custom_hf: any HuggingFace repo in CompVis/LDM format (.ckpt or .safetensors)
custom_hf_repo_id  = "" #@param {type:"string"}
custom_hf_filename = "" #@param {type:"string"}
# e.g. "dreamlike-art/dreamlike-photoreal-2.0", "dreamlike-photoreal-2.0.safetensors"

custom_checkpoint_path = "" #@param {type:"string"}
half_precision = True
check_sha256   = False #@param {type:"boolean"}

# ── Config URLs ────────────────────────────────────────────────────────────────
config_urls = {
    'v1-inference.yaml':
        'https://raw.githubusercontent.com/CompVis/stable-diffusion/main/configs/stable-diffusion/v1-inference.yaml',
    'v2-inference.yaml':
        'https://raw.githubusercontent.com/Stability-AI/stablediffusion/main/configs/stable-diffusion/v2-inference.yaml',
    'v2-inference-v.yaml':
        'https://raw.githubusercontent.com/Stability-AI/stablediffusion/main/configs/stable-diffusion/v2-inference-v.yaml',
}

# ── Model map ──────────────────────────────────────────────────────────────────
# Keys: url, sha256, config, parameterization ('eps'|'v'), default_res
# eps models: SD v1.x, SD 2.x-base  → 512
# v   models: SD 2.x-768            → 768
model_map = {
    "sd-v1-4.ckpt": {
        'sha256': 'fe4efff1e174c627256e44ec2991ba279b3816e364b49f9be2abc0b3ff3f8556',
        'url': 'https://huggingface.co/CompVis/stable-diffusion-v-1-4-original/resolve/main/sd-v1-4.ckpt',
        'requires_login': True, 'config': 'v1-inference.yaml', 'parameterization': 'eps', 'default_res': 512,
    },
    "v1-5-pruned-emaonly.ckpt": {
        'sha256': 'cc6cb27103417325ff94f52b7a5d2dde45a7515b25c255d8e396c90014281516',
        'url': 'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt',
        'requires_login': False, 'config': 'v1-inference.yaml', 'parameterization': 'eps', 'default_res': 512,
    },
    "v2-1_768-ema-pruned.ckpt": {
        'sha256': '',  # verify against HF model card if enabling check_sha256
        'url': 'https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.ckpt',
        'requires_login': False, 'config': 'v2-inference-v.yaml', 'parameterization': 'v', 'default_res': 768,
    },
    "v2-1_512-ema-pruned.ckpt": {
        'sha256': '',
        'url': 'https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.ckpt',
        'requires_login': False, 'config': 'v2-inference.yaml', 'parameterization': 'eps', 'default_res': 512,
    },
    "robo-diffusion-v1.ckpt": {
        'sha256': '244dbe0dcb55c761bde9c2ac0e9b46cc9705ebfe5f1f3a7cc46251573ea14e16',
        'url': 'https://huggingface.co/nousr/robo-diffusion/resolve/main/models/robo-diffusion-v1.ckpt',
        'requires_login': False, 'config': 'v1-inference.yaml', 'parameterization': 'eps', 'default_res': 512,
    },
    "model-epoch05-float16.ckpt": {
        'sha256': '26cf2a2e30095926bb9fd9de0c83f47adc0b442dbfdc3d667d43778e8b70bece',
        'url': 'https://huggingface.co/hakurei/waifu-diffusion-v1-3/resolve/main/model-epoch05-float16.ckpt',
        'requires_login': False, 'config': 'v1-inference.yaml', 'parameterization': 'eps', 'default_res': 512,
    },
}

# ── Resolve path, config, and parameterization ────────────────────────────────
if model_checkpoint == "custom_hf":
    assert custom_hf_repo_id and custom_hf_filename, "Set custom_hf_repo_id and custom_hf_filename"
    print(f"Downloading {custom_hf_filename} from {custom_hf_repo_id}...")
    ckpt_path = huggingface_hub.hf_hub_download(custom_hf_repo_id, custom_hf_filename)
    model_config_key, parameterization, default_res = 'v1-inference.yaml', 'eps', 512
    ckpt_valid = True
elif model_checkpoint == "custom_path":
    ckpt_path = custom_checkpoint_path
    model_config_key, parameterization, default_res = 'v1-inference.yaml', 'eps', 512
    ckpt_valid = Path(ckpt_path).exists()
else:
    entry = model_map[model_checkpoint]
    model_config_key = entry['config']
    parameterization = entry['parameterization']
    default_res      = entry['default_res']
    ckpt_path        = os.path.join(models_path, model_checkpoint)
    ckpt_valid       = True
    if not os.path.exists(ckpt_path):
        url = entry['url']
        if entry.get('requires_login'):
            print("HuggingFace login required for this model.")
            uname = input("Username: "); token = input("Token: ")
            _, path = url.split("https://")
            url = f"https://{uname}:{token}@{path}"
        print(f"Downloading {model_checkpoint}...")
        r = requests.get(url)
        if   r.status_code == 403: raise ConnectionRefusedError("License not accepted.")
        elif r.status_code == 404: raise ConnectionError("Not found at URL.")
        elif r.status_code != 200: raise ConnectionError(f"HTTP {r.status_code}")
        with open(ckpt_path, 'wb') as f: f.write(r.content)
    if check_sha256 and entry.get('sha256'):
        import hashlib
        h = hashlib.sha256(open(ckpt_path,'rb').read()).hexdigest()
        print("SHA256 OK" if h==entry['sha256'] else f"SHA256 MISMATCH (got {h})")

# ── Ensure config YAML ─────────────────────────────────────────────────────────
ckpt_config_path = os.path.join(models_path, model_config_key)
if not os.path.exists(ckpt_config_path):
    sd_cfg = f"stablediffusion/configs/stable-diffusion/{model_config_key}"
    if Path(sd_cfg).exists():
        ckpt_config_path = sd_cfg
    else:
        !wget {config_urls[model_config_key]} -O {ckpt_config_path}

print(f"Config : {ckpt_config_path}  |  Param: {parameterization}  |  Native res: {default_res}px")
print(f"Ckpt   : {ckpt_path}")

# ── Load model ─────────────────────────────────────────────────────────────────
def load_model_from_config(config, ckpt, verbose=False, device='cuda', half_precision=True):
    print(f"Loading model from {ckpt}")
    if ckpt.endswith('.safetensors'):
        from safetensors.torch import load_file
        pl_sd = {"state_dict": load_file(ckpt, device='cpu')}
    else:
        pl_sd = torch.load(ckpt, map_location='cpu')
    if "global_step" in pl_sd: print(f"Global step: {pl_sd['global_step']}")
    model = instantiate_from_config(OmegaConf.load(ckpt_config_path).model)
    m, u = model.load_state_dict(pl_sd["state_dict"], strict=False)
    if verbose:
        if m: print("Missing:", m)
        if u: print("Unexpected:", u)
    return (model.half() if half_precision else model).to(device).eval()

if ckpt_valid:
    local_config = OmegaConf.load(ckpt_config_path)
    sd_model = load_model_from_config(local_config, ckpt_path, half_precision=half_precision)
    sd_model = sd_model.to(device)
    for m in sd_model.modules():
        if hasattr(m,'checkpoint'):     m.checkpoint = False
        if hasattr(m,'use_checkpoint'): m.use_checkpoint = False
    print("Model loaded.")

In [ ]:
#@title Optional: Upgrade VAE (recommended)
use_new_vae = True #@param {type:"boolean"}

if use_new_vae:
    def _hf_dl(repo, fname):
        while True:
            try: return huggingface_hub.hf_hub_download(repo, fname)
            except HTTPError as e:
                if e.response.status_code == 401: huggingface_hub.interpreter_login()
                elif e.response.status_code == 403:
                    input(f"Accept license at https://huggingface.co/{repo} then press Enter: ")
                else: raise

    vae_path = _hf_dl("stabilityai/sd-vae-ft-mse-original", "vae-ft-mse-840000-ema-pruned.ckpt")

    vae_yaml = "stablediffusion/models/first_stage_models/kl-f8/config.yaml"
    if not Path(vae_yaml).exists():
        vae_yaml = "config_vae_kl-f8.yaml"
        if not Path(vae_yaml).exists():
            !wget https://raw.githubusercontent.com/CompVis/latent-diffusion/main/models/first_stage_models/kl-f8/config.yaml -O {vae_yaml}

    vae_cfg = OmegaConf.load(vae_yaml)
    try: vae_cfg['model']['params']['lossconfig']['target'] = "torch.nn.Identity"
    except KeyError: pass
    vae_sd  = torch.load(vae_path, map_location="cpu")["state_dict"]
    vae_m   = instantiate_from_config(vae_cfg.model)
    vae_m.load_state_dict(vae_sd, strict=False)
    vae_m   = vae_m.half().to(device).eval().requires_grad_(False)
    del sd_model.first_stage_model
    sd_model.first_stage_model = vae_m
    print("VAE upgraded.")

In [ ]:
#@title Settings

n      = 300  #@param {type:"integer"}

# Leave at 0 to auto-set from the loaded model's native resolution (512 or 768)
height = 0    #@param {type:"integer"}
width  = 0    #@param {type:"integer"}
if height == 0: height = default_res
if width  == 0: width  = default_res
assert height % 8 == 0 and width % 8 == 0
print(f"Resolution: {width}x{height}")

seed = -1  #@param {type:"number"}

# Path to an init image, or leave blank
init_image = ""  #@param {type:'string'}

# Perlin noise: neutral structured init with maximum freedom of movement.
# Used when init_image is blank. Falls back to Gaussian if 'noise' package missing.
use_perlin_init = True  #@param {type:"boolean"}

# Keyframe curves: "t:v, t:v, ..." — values interpolated with smooth S-curves
g     = "0:0.09"  #@param {type:"string"}
sigma = "2.25"    #@param {type:"string"}
h     = ".1"      #@param {type:"string"}
gamma = "1.1"     #@param {type:"string"}
alpha = "0.005"   #@param {type:"string"}
tau   = "1.0"     #@param {type:"string"}
k     = .07       #@param {type:"number"}

refinement_steps = "6"  #@param {type:"string"}
img_init_steps   = 15   #@param {type:"number"}

hvp_method = 'kronecker'  #@param ["forward-functorch", "reverse", "fake", "zero", "kronecker"]

In [ ]:
#@title Prompts
# [text, {time: weight, ...}]  Negative weights push away from a concept.
# First prompt initializes the image when no init_image is provided.

prompt_params = [
    ["incredibly beautiful orchids, a bouquet of orchids",      {0:1,     35:1,    50:0}],
    ["incredibly beautiful roses, a bouquet of roses",          {0:0.001, 35:0.001, 50:1, 120:1,  140:0}],
    ["incredibly beautiful carnations, a bouquet of carnations",{0:0.001, 120:0.001,140:1, 220:1,  240:0}],
    ["incredibly beautiful sunflowers, a bouquet of sunflowers",{0:0.001, 220:0.001,240:1}],
    ["watermark text",                       {0:-0.1}],
    ["jpeg artifacts",                       {0:-0.1}],
    ["artist's signature",                  {0:-0.1}],
    ["istockphoto, gettyimages, watermark",  {0:-0.1}],
]

In [ ]:
#@title Build Prompts & Settings, then Generate

resume           = False  #@param {type:'boolean'}
archive_old_work = False  #@param {type:'boolean'}
resume_from      = -1     #@param {type:'number'}

plot_prompt_weights = False  #@param {type:'boolean'}
plot_h = plot_g = plot_sigma = plot_gamma = plot_alpha = plot_tau = False

prompts = [Prompt(t, ws) for t, ws in prompt_params]

curved_settings = ParameterGroup({
    'g':     SmoothCurve(parse_curvable_string(g)),
    'sigma': SmoothCurve(parse_curvable_string(sigma)),
    'h':     SmoothCurve(parse_curvable_string(h)),
    'gamma': SmoothCurve(parse_curvable_string(gamma)),
    'alpha': SmoothCurve(parse_curvable_string(alpha)),
    'tau':   SmoothCurve(parse_curvable_string(tau)),
    'steps': SmoothCurve(parse_curvable_string(refinement_steps)),
    'k':     SmoothCurve(parse_curvable_string(k)),
})

_seed = seed if seed >= 0 else random.randrange(0, 4294967295)
print(f"Seed: {_seed}")
torch.manual_seed(_seed)

stuff_to_plot = [name for flag, name in [
    (plot_prompt_weights,'prompts'),(plot_h,'h'),(plot_g,'g'),
    (plot_sigma,'sigma'),(plot_gamma,'gamma'),(plot_alpha,'alpha'),(plot_tau,'tau')
] if flag]

if not resume:
    if archive_old_work:
        adir = outdir.parent / 'archive' / str(int(time.time()))
        adir.mkdir(parents=True, exist_ok=True)
        for p in outdir.glob('*'): p.rename(adir/p.name)
    else:
        for p in outdir.glob('*'): p.unlink()
        for p in debug_dir.glob('*'): p.unlink()

_init = init_image or None
if not _init and use_perlin_init:
    print("Generating Perlin init...")
    _init = generate_perlin_init(height, width, seed=_seed)

sample_mcmc_klmc2(
    sd_model=sd_model, init_image=_init,
    height=height, width=width, n=n, hvp_method=hvp_method,
    prompts=prompts, settings=curved_settings,
    resume=resume, resume_from=resume_from,
    img_init_steps=img_init_steps, stuff_to_plot=stuff_to_plot,
)

In [ ]:
#@title Make Video
embed_video    = True   #@param {type:'boolean'}
download_video = False  #@param {type:'boolean'}
upscale_video  = False  #@param {type:'boolean'}
fps       = 14          #@param {type:"integer"}
out_fname = "out.mp4"   #@param {type:"string"}

out_fullpath = str(outdir / out_fname)
scale_arg = f"-vf scale={2*width}x{2*height}:flags=lanczos " if upscale_video else ""
!cd {str(outdir)}; ffmpeg -y -r {fps} -i 'out_%*.png' -crf 15 -preset veryslow -pix_fmt yuv420p {scale_arg}{out_fname}

if embed_video: show_video(out_fullpath)
if download_video and probably_using_colab:
    from google.colab import files
    files.download(out_fullpath)